# ☁️ Cloud Type Classification: A Deep Learning Approach

## 📌 Project Overview
Atmospheric scientists and meteorologists rely heavily on cloud classification to understand weather patterns, climate change, and hydrological cycles. The World Meteorological Organization (WMO) classifies clouds into 10 major genera based on their altitude and visual characteristics.

In this project, we aim to automate the process of classifying ground-based sky images into **11 different cloud categories** (including contrails) using two approaches:
1. A **custom CNN trained from scratch** as our baseline.
2. A **Transfer Learning model (EfficientNetB0)** pretrained on ImageNet as our improved model.

## 📊 Dataset: Cirrus Cumulus Stratus Nimbus (CCSN) Database
The CCSN dataset contains **2,543 ground-based sky images** across 11 WMO-defined cloud genus categories. Cloud structures are highly amorphous and exhibit immense intra-class variation while sharing significant inter-class similarities — making this a genuinely challenging computer vision task.

### 🗂️ Class Dictionary
| Code | Name | Description |
|------|------|-------------|
| Ac | Altocumulus | Mid-level patchy clouds, often in rolls or ripples |
| As | Altostratus | Grayish/bluish cloud sheets covering the sky |
| Cb | Cumulonimbus | Towering storm clouds associated with thunderstorms |
| Cc | Cirrocumulus | High-altitude, small white patches of clouds |
| Ci | Cirrus | Delicate, feathery, high-altitude ice clouds |
| Cs | Cirrostratus | Transparent whitish veils causing halos |
| Ct | Contrail | Artificial clouds created by aircraft exhaust |
| Cu | Cumulus | Puffy, cotton-like low clouds with flat bases |
| Ns | Nimbostratus | Dark, low-level rain-producing clouds |
| Sc | Stratocumulus | Low, lumpy layers or patches of clouds |
| St | Stratus | Uniform, featureless, gray low-level clouds |

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Conv2D, MaxPooling2D, Flatten,
                                     Dense, Dropout, GlobalAveragePooling2D)
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.utils import to_categorical

import warnings
warnings.filterwarnings('ignore')


## 1. Global Configurations & Setup
We define global hyperparameters used throughout the entire pipeline. We use **224×224** — the ImageNet standard resolution — as it preserves cloud texture detail while remaining compatible with our EfficientNetB0 backbone and manageable within Colab's free-tier VRAM.

In [ ]:

IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 15
DATASET_PATH = 'dataset/CCSN_v2'

# Strict hardcoded class order - must stay consistent across the entire pipeline
# INJECT BUG 1: Index Mismatch Across Pipeline here
CLASS_NAMES = ['Ac', 'As', 'Cb', 'Cc', 'Ci', 'Cs', 'Ct', 'Cu', 'Ns', 'Sc', 'St']
NUM_CLASSES = len(CLASS_NAMES)

print(f"Tracking {NUM_CLASSES} cloud categories: {CLASS_NAMES}")

##  2. Data Ingestion Pipeline
We traverse the dataset directory structure, loading images via OpenCV and mapping each folder name to its ground-truth class index. OpenCV reads images in BGR by default — we convert to RGB so visualizations and model inputs are colour-correct.

In [ ]:
# Upload dataset zip
from google.colab import files
import zipfile

uploaded = files.upload()
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('dataset/')

def load_cloud_dataset(data_path, class_names, img_size):
    images, labels = [], []
    folder_names = sorted(os.listdir(data_path))

    for folder in folder_names:
        folder_path = os.path.join(data_path, folder)
        if not os.path.isdir(folder_path):
            continue

        # INJECT BUG 2: Folder Alphabetical Order vs. Hardcoded Label Mismatch
        if folder in class_names:
            class_idx = class_names.index(folder)

            for img_name in os.listdir(folder_path):
                img_path = os.path.join(folder_path, img_name)
                img = cv2.imread(img_path)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, img_size)
                images.append(img)
                labels.append(class_idx)

    return np.array(images), np.array(labels)

# Execute the data loader
X_clean, y_clean = load_cloud_dataset(DATASET_PATH, CLASS_NAMES, IMG_SIZE)
X_clean = X_clean.astype('float32') / 255.0
print(f"Successfully loaded {len(X_clean)} images into memory.")


##  3. Stratified Data Splitting
We partition into three sets: **70% train / 15% validation / 15% test**. Stratified sampling ensures every cloud type appears proportionally in each split — critical when some classes (e.g. Ci, Cu) have significantly fewer samples than others.

In [ ]:

# INJECT BUG 3: Split/stratification ordering issue here

# Primary split — 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X_clean, y_clean,
    test_size=0.3,
    random_state=42,
    shuffle=True,
    stratify=y_clean
)

# Secondary split — split temp 50/50 into val and test (15% each overall)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    shuffle=True,
    stratify=y_temp
)

print(f"Train:      {len(X_train)} samples")
print(f"Validation: {len(X_val)} samples")
print(f"Test:       {len(X_test)} samples")

# One-hot encode for categorical cross-entropy
y_train_cat = to_categorical(y_train, num_classes=NUM_CLASSES)
y_val_cat   = to_categorical(y_val,   num_classes=NUM_CLASSES)
y_test_cat  = to_categorical(y_test,  num_classes=NUM_CLASSES)

##  4. Data Augmentation
CCSN contains only ~2,543 images — far too few to train a robust CNN from scratch without overfitting. Data augmentation synthetically expands the effective training set by applying randomized but realistic transformations at runtime.

In [ ]:
# Training generator — augmented
train_datagen = ImageDataGenerator(
    horizontal_flip=True,
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    zoom_range=0.05
)

# Val/Test generators — no augmentation, raw normalized data only
val_datagen  = ImageDataGenerator()

train_gen = train_datagen.flow(X_train, y_train_cat, batch_size=BATCH_SIZE, shuffle=True)
val_gen   = val_datagen.flow(X_val,   y_val_cat,   batch_size=BATCH_SIZE, shuffle=False)

print("Augmentation generators configured.")

## 5. Baseline: Custom CNN (Trained from Scratch)
Our baseline is a lightweight 3-block CNN trained entirely on CCSN data.

In [ ]:
def build_scratch_cnn(input_shape, num_classes):
    model = Sequential([
        Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        MaxPooling2D((2, 2)),

        Conv2D(64, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),

        Conv2D(128, (3, 3), activation='relu', padding='same'),
        MaxPooling2D((2, 2)),

        GlobalAveragePooling2D(),
        Dropout(0.5),
        Dense(num_classes, activation='softmax')
    ])
    return model

scratch_model = build_scratch_cnn((*IMG_SIZE, 3), NUM_CLASSES)
scratch_model.summary()

##  6. Compile & Train - Scratch CNN


We use `categorical_crossentropy` — the mathematically correct loss for multi-class classification with one-hot encoded targets.

In [ ]:

# INJECT BUG 4: Wrong Loss Function here

scratch_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Compute class weights
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))

early_stop = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

scratch_history = scratch_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=1
)

##  7. Evaluation - Scratch CNN
We evaluate against the held-out test set.

In [ ]:
scratch_loss, scratch_acc = scratch_model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Scratch CNN — Test Accuracy: {scratch_acc:.4%}  |  Test Loss: {scratch_loss:.4f}\n")

y_pred_scratch = np.argmax(scratch_model.predict(X_test), axis=1)

cm_scratch = confusion_matrix(y_test, y_pred_scratch)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_scratch, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Scratch CNN — Confusion Matrix', pad=15)
plt.ylabel('Ground Truth')
plt.xlabel('Predicted')
plt.show()

##  8. Improved Model: Transfer Learning with EfficientNetB0
Training from scratch on limited images often causes underperformance. **Transfer learning** sidesteps this by reusing an EfficientNetB0 backbone pretrained on 1.2M ImageNet images.

> *EfficientNetB0 expects inputs in the range [0, 255] — so we do **not** pass the pre-normalized `[0, 1]` arrays. Instead, we pass the raw pixel arrays and let the built-in `preprocess_input` layer handle normalization internally.*

In [ ]:
from tensorflow.keras.applications.efficientnet import preprocess_input

# INJECT BUG 5: Preprocessing Mismatch here

X_train_eff = preprocess_input(X_train * 255.0)
X_val_eff   = preprocess_input(X_val   * 255.0)
X_test_eff  = preprocess_input(X_test  * 255.0)

train_gen_eff = train_datagen.flow(X_train_eff, y_train_cat, batch_size=BATCH_SIZE, shuffle=True)
val_gen_eff   = val_datagen.flow(X_val_eff,   y_val_cat,   batch_size=BATCH_SIZE, shuffle=False)

backbone = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(*IMG_SIZE, 3)
)

# INJECT BUG 6: Catastrophic Forgetting (Forgetting to freeze backbone) here

backbone.trainable = False

# Attach custom classification head
x = backbone.output

# INJECT BUG 7: Dimension Mismatch / Missing Flatten here

x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

tl_model = Model(inputs=backbone.input, outputs=output)
tl_model.summary()

## 9. Compile & Train - Transfer Learning Model

In [ ]:
class_weights_array = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights_array))

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
early_stop_tl = EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True)

tl_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

tl_history = tl_model.fit(
    train_gen_eff,
    validation_data=val_gen_eff,
    epochs=10,
    class_weight=class_weight_dict,
    callbacks=[reduce_lr, early_stop_tl],
    verbose=1
)


##  10. Evaluation - Transfer Learning Model

In [ ]:
tl_loss, tl_acc = tl_model.evaluate(X_test_eff, y_test_cat, verbose=0)
print(f"Transfer Learning — Test Accuracy: {tl_acc:.4%}  |  Test Loss: {tl_loss:.4f}\n")
print(f"Improvement over Scratch CNN: +{(tl_acc - scratch_acc):.4%}")

y_pred_tl = np.argmax(tl_model.predict(X_test_eff), axis=1)


# Write the confusion matrix and classification report for the transfer-learning model here — mirror what you did for the scratch CNN above.
cm_tl = confusion_matrix(y_test, y_pred_tl)

plt.figure(figsize=(12, 10))
sns.heatmap(cm_tl, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            linewidths=0.5, linecolor='white')
plt.title('EfficientNetB3 Transfer Learning — Confusion Matrix', fontsize=15, pad=15)
plt.ylabel('Ground Truth')
plt.xlabel('Predicted')
plt.xticks(rotation=45)
plt.show()

print("\n" + "="*60)
print("TRANSFER LEARNING — CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_test, y_pred_tl, target_names=CLASS_NAMES, zero_division=0))